In [ ]:
from ugot import ugot  # UGOT robot SDK
import time  # For sleep/timing between motor commands
import importlib
import cv2
import numpy as np
from IPython.display import display, clear_output, Image
from PIL import Image as Image2

# Create a robot instance and connect over the network.
got = ugot.UGOT()
got.initialize("192.168.88.1")  # <-- Check if this matches your robot's IP address

# Pre-load vision models onto the UGOT
# All models listed here will be available for the rest of the session.
got.load_models(
    ["color_recognition", "word_recognition", "line_recognition", "apriltag_qrcode", "face_recognition"]
)

# Open UGOT camera for later image recognition
got.open_camera()

Seen = False
Reached = False

def face_find_and_approach(gap, target_name, turn_spd, strafe_spd, fwd_spd, height):
    global Seen, Reached
    # Phase 1 - Find
    face_name = None


    while not Seen:
        got.mecanum_turn_speed(turn=3, speed=turn_spd)
        name = got.get_words_result()
        faces = got.get_face_recognition_total_info()

        if faces:
            for face in faces:
                if face[0] == target_name:
                    Seen = True
                    got.mecanum_stop()
                    print(f"Saw {target_name}!")
                    return

    # Phase 2 - Move
    while Seen and not Reached:
        name = got.get_words_result()
        faces = got.get_face_recognition_total_info()
        if not faces:
            # Lost the face; inch forward slowly to try to find it again
            got.mecanum_translate_speed(angle=0, speed=fwd_spd)
        else:
            c_x = faces[0][1]  # Horizontal center of the face in the frame (0–640 px)
            h = faces[0][3]  # Height of the face bounding box (proxy for distance)
            if h < height:
                if c_x < 320 - gap:
                    # Face is too far LEFT — strafe left while moving forward
                    got.mecanum_move_xyz(
                        x_speed=-strafe_spd, y_speed=fwd_spd, z_speed=0
                    )
                elif c_x > 320 + gap:
                    # Face is too far RIGHT — strafe right while moving forward
                    got.mecanum_move_xyz(x_speed=strafe_spd, y_speed=fwd_spd, z_speed=0)
                else:
                    # Face is centered but still small (far away) — move straight forward
                    got.mecanum_move_xyz(x_speed=0, y_speed=fwd_spd, z_speed=0)
            else:
                # Face is centered AND large enough — we've arrived!
                got.mecanum_stop()
                print(f"Reached {name}!")
                Reached = True
        # clear_output(wait=True)

    if Seen and Reached:
        got.mecanum_stop()
        return True

try:
    while True:
        got.mechanical_clamp_close()
        got.mechanical_joint_control(angle1=0, angle2=90, angle3=-90, duration=500)
        AtTarget = face_find_and_approach(gap=15, target_name="Arjun", turn_spd=15, strafe_spd=10, fwd_spd=10, height=120)
        if AtTarget:
            got.mecanum_translate_speed_times(angle=180, speed=10, times=10, unit=1)
            time.sleep(0.5)
            got.mechanical_joint_control(angle1=0, angle2=0, angle3=-60, duration=500)
            time.sleep(1)
            got.mechanical_clamp_release()
            got.mecanum_translate_speed_times(angle=180, speed=10, times=5, unit=1)
            break
except KeyboardInterrupt:
    got.mecanum_stop()
    print("Done.")

192.168.88.1:50051
Saw Arjun!
Reached Arju!
